# Fake News Detection — Phase 1: Data Cleaning

**Project goal:** Build an AI model that reads a news article (title + body text) and predicts
whether it is `Fake News` or `Factual News`.

**This notebook = Phase 1 only.** Before we can train any model, we have to take the raw,
messy CSV and turn it into a clean, well-structured dataset. Phase 2 (later notebook) will
take the output of this notebook and actually train/evaluate a classifier (Logistic Regression,
like we practiced in the course's `02-Logistic_Regression_Project`).

**Techniques used below are pulled directly from the course notebooks:**
- `df.info()`, `df.describe()`, `df.head()` → from `02-Logistic_Regression_Project`, `03-DataFrames`
- `df.isnull().sum()`, `df.dropna()`, `df.fillna()` → from `04-Missing_Data`
- `df.duplicated()` → standard pandas DataFrame method covered alongside `03-DataFrames`
- `.str` string methods (`.strip()`, `.lower()`, `.split()`) → from `Python_Exercises_-_Solutions`
  (e.g. the `s.split()` and `domainGet()` exercises) and `03-DataFrames`
- `.apply()` with a custom function → from `03-DataFrames` / `05-Groupby`
- writing small custom functions (`def ...:`) → from `Lab` and `Python_Exercises_-_Solutions`
  (e.g. `welcome()`, `domainGet()`, `findDog()`, `countDog()`, `caught_speeding()`)
- `lambda` + filtering pattern → from the `filter(lambda ...)` exercise in `Python_Exercises_-_Solutions`
- NumPy (`np.nan`, vectorized array ops) → from `Numpy (with comments)`
- `pd.to_datetime()` → standard pandas date handling, same family as the `Operations`/`DataFrames` notebooks

**What goes beyond the course (and why it's still necessary):**
This is a **text/NLP** dataset, not a numeric one like the course's salary or advertising datasets.
So on top of the course techniques, we add a few NLP-specific cleaning steps (lowercasing,
punctuation/stopword removal) that weren't covered in the Pandas/Matplotlib lectures but are
required before any text can be fed into a machine learning model. These are clearly marked
below as **[NLP STEP]** so you know which parts are 'new' versus which parts are course material.

## 0. Imports

We import `pandas` for our DataFrame, `numpy` for handling `NaN`/numeric edge cases (as in the
Missing Data and Numpy notebooks), and Python's built-in `re` (regular expressions) and `string`
modules for the text-cleaning step. No external NLP library (like nltk) is required — we keep
this self-contained so it runs anywhere without extra downloads.

In [1]:
# Core libraries used throughout the course (Pandas + Numpy sections)
import pandas as pd
import numpy as np

# Built-in Python modules for text cleaning (no extra installs needed)
import re       # regular expressions: used to strip URLs, digits, extra whitespace
import string   # gives us string.punctuation, the list of standard punctuation characters


## 1. Load the Data & First Look

Same first move as every course project (`SF_Salaries`, `Linear_Regression_Project`,
`Logistic_Regression_Project`): read the CSV, then immediately check `.head()`, `.info()`,
and `.describe()` to understand what we're working with before touching anything.

In [ ]:
# Read the raw CSV file into a DataFrame — exactly like ad_data = pd.read_csv('advertising.csv')
# in the Logistic Regression Project notebook.
df = pd.read_csv('fake_news_data.csv')

# Peek at the first 5 rows to see what the columns look like
df.head()

,title,text,date,fake_or_factual
0,HOLLYWEIRD LIB SUSAN SARANDON Compares Muslim ...,There are two small problems with your analogy...,"Dec 30, 2015",Fake News
1,Elijah Cummings Called Trump Out To His Face ...,Buried in Trump s bonkers interview with New Y...,"April 6, 2017",Fake News
2,Hillary Clinton Says Half Her Cabinet Will Be...,"Women make up over 50 percent of this country,...","April 26, 2016",Fake News
3,Russian bombing of U.S.-backed forces being di...,WASHINGTON (Reuters) - U.S. Defense Secretary ...,"September 18, 2017",Factual News
4,Britain says window to restore Northern Irelan...,BELFAST (Reuters) - Northern Ireland s politic...,"September 4, 2017",Factual News


In [3]:
# .info() tells us: column names, non-null counts, and data types.
# This is our first checkpoint for spotting missing data (just like in 04-Missing_Data).
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 198 entries, 0 to 197
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   title            198 non-null    object
 1   text             198 non-null    object
 2   date             198 non-null    object
 3   fake_or_factual  198 non-null    object
dtypes: object(4)
memory usage: 6.3+ KB


In [4]:
# .shape tells us (rows, columns) — quick sanity check on dataset size
print('Rows, Columns:', df.shape)

# .columns shows us exactly what fields we have to work with
print('Columns:', df.columns.tolist())

Rows, Columns: (198, 4)
Columns: ['title', 'text', 'date', 'fake_or_factual']


## 2. Check for Missing Data

Direct application of the **`04-Missing_Data`** notebook's technique: `df.isnull().sum()` to count
missing values per column, then decide whether to `dropna()`, `fillna()`, or — as in this case —
do nothing because there isn't any missing data.

In [5]:
# Count how many missing (NaN) values exist in each column
# Same line of code used in the Missing Data lecture notebook
df.isnull().sum()

title              0
text               0
date               0
fake_or_factual    0
dtype: int64

**Result:** all four columns show `0` missing values. This dataset happens to be clean on this
front, so we don't need `df.dropna()` here — but it's still good practice to run this check on
every project before moving forward, since assuming there's no missing data is a common source
of bugs later on.

**[NLP STEP]** Beyond `isnull()`, text data can also be 'secretly missing' as an *empty string*
(`''`) rather than `NaN` — a CSV could contain a blank field that pandas doesn't flag as null.
We check for that too, since `.isnull()` alone wouldn't catch it.

In [6]:
# Strip whitespace first (a cell that's just spaces should count as 'empty')
# then check how many rows are empty strings in each column
for col in df.columns:
    empty_count = (df[col].astype(str).str.strip() == '').sum()
    print(f"{col}: {empty_count} empty-string rows")

title: 0 empty-string rows
text: 0 empty-string rows
date: 0 empty-string rows
fake_or_factual: 0 empty-string rows


## 3. Check for Duplicate Rows

Duplicate articles would let the model 'see' the same example more than once, which can bias
training and make evaluation misleading. We check both full-row duplicates and duplicates based
only on `title`+`text` (in case the same article appears with a different date, for example).

In [7]:
# Check for fully duplicated rows (every column identical)
print('Fully duplicated rows:', df.duplicated().sum())

# Check for duplicated articles based on title + text only
# (in case the same article was logged twice with different dates/labels)
print('Duplicate title+text rows:', df.duplicated(subset=['title', 'text']).sum())

Fully duplicated rows: 0
Duplicate title+text rows: 0


**Result:** 0 duplicates either way. If duplicates had existed, the course-style fix would be:
`df = df.drop_duplicates(subset=['title','text'])` — same idea as filtering a DataFrame in
`03-DataFrames` / `07-Operations`.

## 4. Strip Extra Whitespace

Several `title` and `date` values have leading/trailing spaces (e.g. `' Elijah Cummings...'`).
This is the same `.str.strip()` pattern used on strings throughout the Python exercises
(e.g. cleaning up `s = 'Hi there Sam!'` before splitting it). Left uncleaned, these spaces can
cause `'Fake News'` and `' Fake News'` to be treated as two different categories, or break exact
string matching later on.

In [8]:
# Apply .str.strip() to every text column to remove leading/trailing whitespace
# This is the same .strip() idea from the Python Crash Course string exercises
text_columns = ['title', 'text', 'date', 'fake_or_factual']

for col in text_columns:
    df[col] = df[col].astype(str).str.strip()

# Confirm it worked: this should now print 0 for every column
for col in text_columns:
    leftover_spaces = (df[col] != df[col].str.strip()).sum()
    print(f"{col}: {leftover_spaces} rows with leftover whitespace")

title: 0 rows with leftover whitespace
text: 0 rows with leftover whitespace
date: 0 rows with leftover whitespace
fake_or_factual: 0 rows with leftover whitespace


## 5. Inspect & Encode the Label Column

`fake_or_factual` is our **target variable** — what the model will learn to predict. Machine
learning models need numbers, not text labels, so just like in the Logistic Regression project
(where categorical features get mapped to 0/1), we convert `'Fake News'` / `'Factual News'` into
a numeric `label` column.

In [9]:
# See exactly what unique values exist in the label column, and how balanced the classes are.
# A roughly even split (no class massively outnumbering the other) is good — it means accuracy
# will be a meaningful metric later, rather than misleading due to class imbalance.
df['fake_or_factual'].value_counts()

fake_or_factual
Factual News    100
Fake News        98
Name: count, dtype: int64

In [10]:
# Map the text labels to numbers: 1 = Fake News, 0 = Factual News.
# This 'label' column is what our Phase 2 model will actually try to predict.
df['label'] = df['fake_or_factual'].map({'Fake News': 1, 'Factual News': 0})

# Sanity check: confirm there are no NaNs introduced by the mapping
# (a NaN here would mean some label spelling wasn't 'Fake News' or 'Factual News' exactly)
print('Unmapped/NaN labels:', df['label'].isnull().sum())
df[['fake_or_factual', 'label']].drop_duplicates()

Unmapped/NaN labels: 0


,fake_or_factual,label
0,Fake News,1
3,Factual News,0


## 6. Parse the Date Column

The raw `date` column mixes two formats — full month names (`'September 18, 2017'`) and
abbreviated ones (`'Dec 30, 2015'`) — plus inconsistent trailing spaces (already fixed in
Section 4). We convert this text column into an actual datetime type using `pd.to_datetime()`,
which is the same function family used for date handling in the course's pandas notebooks.
`format='mixed'` tells pandas to figure out each row's format individually, since they're not
all the same.

In [11]:
# Convert the date strings into real datetime objects.
# errors='coerce' means: if a date truly can't be parsed, turn it into NaT (missing) instead
# of crashing — this mirrors the 'fail gracefully, then check for missing data' approach from
# the Missing Data notebook.
df['date'] = pd.to_datetime(df['date'], format='mixed', errors='coerce')

# Check whether any dates failed to parse
print('Unparseable dates (NaT):', df['date'].isnull().sum())

df['date'].head()

Unparseable dates (NaT): 0


0   2015-12-30
1   2017-04-06
2   2016-04-26
3   2017-09-18
4   2017-09-04
Name: date, dtype: datetime64[ns]

In [12]:
# Now that 'date' is a real datetime column, we can pull out useful features from it,
# the same way you'd derive new columns in the GroupBy/Operations notebooks.
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month

df[['date', 'year', 'month']].head()

,date,year,month
0,2015-12-30,2015,12
1,2017-04-06,2017,4
2,2016-04-26,2016,4
3,2017-09-18,2017,9
4,2017-09-04,2017,9


## 7. [NLP STEP] Clean the Article Text

This is the part that goes beyond the Pandas/Matplotlib course material, because `title` and
`text` are **free-form natural language**, not numbers or clean categories. A machine learning
model can't directly understand raw sentences — we need to normalize the text first.

We write our own `clean_text()` function (same pattern as `domainGet()`, `findDog()`, and
`countDog()` from the Python exercises notebook: define a function, do some string processing
inside it, return the result) that does the following, in order:

1. **Lowercase everything** — so `'Trump'` and `'trump'` are treated as the same word.
2. **Remove URLs and emails** — leftover web addresses don't carry meaningful content for the model.
3. **Remove HTML tags** — in case any `<...>` markup slipped into the scraped text.
4. **Remove punctuation** — including 'smart quotes' (`’ ‘ “ ”`) which showed up in the titles
   and aren't covered by Python's standard `string.punctuation`.
5. **Remove standalone digits** — numbers rarely help a bag-of-words style model distinguish
   fake vs. factual news, and they add noise.
6. **Collapse extra whitespace** left behind by the steps above.
7. **Remove stopwords** — common filler words (`'the'`, `'is'`, `'and'`, etc.) that appear in
   almost every sentence and don't help discriminate between classes.
8. **Drop very short tokens** (length 1) — leftover single characters from the cleanup.

**Note on a data-quality quirk we found:** several articles have missing apostrophes in the raw
text itself (e.g. `"who s not paying attention"` instead of `"who's not paying attention"`).
This looks like an artifact from however the original dataset was scraped/exported — it is NOT
something introduced by our cleaning. We don't attempt to guess and reinsert the missing
apostrophes (that would risk introducing errors), and our cleaning function handles it gracefully
either way since punctuation gets stripped regardless.

In [13]:
# Standard English stopwords — common words that carry little distinguishing meaning on their own.
# We define this list manually (instead of relying on an external NLTK download) to keep the
# notebook fully self-contained and runnable offline.
STOPWORDS = set("""
a about above after again against all am an and any are aren't as at be because been
before being below between both but by can't cannot could couldn't did didn't do does
doesn't doing don't down during each few for from further had hadn't has hasn't have
haven't having he he'd he'll he's her here here's hers herself him himself his how
how's i i'd i'll i'm i've if in into is isn't it it's its itself let's me more most
mustn't my myself no nor not of off on once only or other ought our ours ourselves out
over own same shan't she she'd she'll she's should shouldn't so some such than that
that's the their theirs them themselves then there there's these they they'd they'll
they're they've this those through to too under until up very was wasn't we we'd we'll
we're we've were weren't what what's when when's where where's which while who who's
whom why why's with won't would wouldn't you you'd you'll you're you've your yours
yourself yourselves
""".split())

# Characters that string.punctuation misses: 'smart quotes' and long dashes that appear
# in the scraped article titles (e.g. the curly apostrophe in "Jesus' Family").
UNICODE_PUNCT = '\u2019\u2018\u201c\u201d\u2014\u2013\u2026'
ALL_PUNCT = string.punctuation + UNICODE_PUNCT


def clean_text(text):
    """
    Take a raw string of article text/title and return a cleaned,
    lowercased, stopword-free version ready for an NLP model.
    Follows the same 'define a function, return a value' pattern
    as the custom functions in the Python exercises notebook.
    """
    # Step 1: lowercase
    text = text.lower()

    # Step 2: remove URLs and email addresses
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)

    # Step 3: remove any leftover HTML tags
    text = re.sub(r'<.*?>', ' ', text)

    # Step 4: remove punctuation (including curly quotes/dashes)
    text = text.translate(str.maketrans('', '', ALL_PUNCT))

    # Step 5: remove standalone digits
    text = re.sub(r'\d+', ' ', text)

    # Step 6: collapse multiple spaces into one, and trim the ends
    text = re.sub(r'\s+', ' ', text).strip()

    # Step 7 & 8: remove stopwords and single-character leftovers
    tokens = [word for word in text.split() if word not in STOPWORDS and len(word) > 1]

    return ' '.join(tokens)

In [14]:
# Apply our custom clean_text() function to both the title and the article body.
# .apply() on a Series runs the function on every row — same pattern used in
# 03-DataFrames and 05-Groupby when applying custom functions across a column.
df['clean_title'] = df['title'].apply(clean_text)
df['clean_text'] = df['text'].apply(clean_text)

# Compare a raw title to its cleaned version, side by side
df[['title', 'clean_title']].head()

,title,clean_title
0,HOLLYWEIRD LIB SUSAN SARANDON Compares Muslim ...,hollyweird lib susan sarandon compares muslim ...
1,Elijah Cummings Called Trump Out To His Face A...,elijah cummings called trump face heres pathet...
2,Hillary Clinton Says Half Her Cabinet Will Be ...,hillary clinton says half cabinet will women e...
3,Russian bombing of U.S.-backed forces being di...,russian bombing usbacked forces discussed high...
4,Britain says window to restore Northern Irelan...,britain says window restore northern ireland d...


In [15]:
# Spot-check: compare the first 200 characters of raw vs. cleaned article body
print('RAW:\n', df['text'].iloc[0][:200])
print()
print('CLEANED:\n', df['clean_text'].iloc[0][:200])

RAW:
 There are two small problems with your analogy Susan  Jesus was NOT a Muslim and Joseph traveled to Bethlehem with Mary. For anyone who s not paying attention there don t appear to be many female refu

CLEANED:
 two small problems analogy susan jesus muslim joseph traveled bethlehem mary anyone paying attention don appear many female refugees accompanying male muslim refugees greecesusan sarandon spent christ


In [16]:
# Safety check: make sure cleaning didn't accidentally wipe any article down to nothing
empty_after_cleaning = (df['clean_text'].str.len() == 0).sum()
print('Rows with empty clean_text after cleaning:', empty_after_cleaning)

Rows with empty clean_text after cleaning: 0


## 8. Add a Few Simple Derived Features

While we're here, we add some lightweight numeric features that are cheap to compute and can
be useful later (fake news articles are sometimes longer/shorter, or written in ALL CAPS more
often, etc.). This is the same idea as creating new columns from existing ones in
`07-Operations` (e.g. deriving a new column from an existing one with `.apply()` or vectorized math).

In [17]:
# Length of the raw article (characters) and cleaned word count —
# simple numeric signals that might correlate with fake vs. factual news
df['text_length'] = df['text'].apply(len)
df['word_count'] = df['clean_text'].apply(lambda x: len(x.split()))

# What fraction of the ORIGINAL title is uppercase letters —
# clickbait-y fake news headlines are often written in ALL CAPS
def uppercase_ratio(title):
    letters = [ch for ch in title if ch.isalpha()]
    if len(letters) == 0:
        return 0.0
    upper_letters = [ch for ch in letters if ch.isupper()]
    return len(upper_letters) / len(letters)

df['title_upper_ratio'] = df['title'].apply(uppercase_ratio)

df[['text_length', 'word_count', 'title_upper_ratio']].describe()

,text_length,word_count,title_upper_ratio
count,198.000000,198.000000,198.000000
mean,2283.560606,214.025253,0.219104
std,1899.952197,174.840709,0.192361
min,74.000000,5.000000,0.018182
25%,995.000000,93.250000,0.065574
50%,2072.000000,191.000000,0.135801
75%,2892.250000,270.750000,0.343333
max,19130.000000,1747.000000,1.000000


## 9. Final Review of the Cleaned Dataset

Reorder and review the columns one last time, the same way every course notebook ends its
cleaning section with a final `.head()` / `.info()` check before moving on to analysis or modeling.

In [18]:
# Reorder columns so the dataset reads logically: identifiers first, then cleaned text,
# then engineered features, then the label last (label is the model's target).
final_columns = [
    'title', 'clean_title',
    'text', 'clean_text',
    'date', 'year', 'month',
    'text_length', 'word_count', 'title_upper_ratio',
    'fake_or_factual', 'label'
]
df = df[final_columns]

df.head()

,title,clean_title,text,clean_text,date,year,month,text_length,word_count,title_upper_ratio,fake_or_factual,label
0,HOLLYWEIRD LIB SUSAN SARANDON Compares Muslim ...,hollyweird lib susan sarandon compares muslim ...,There are two small problems with your analogy...,two small problems analogy susan jesus muslim ...,2015-12-30,2015,12,1992,182,0.524590,Fake News,1
1,Elijah Cummings Called Trump Out To His Face A...,elijah cummings called trump face heres pathet...,Buried in Trump s bonkers interview with New Y...,buried trump bonkers interview new york times ...,2017-04-06,2017,4,2836,251,0.233766,Fake News,1
2,Hillary Clinton Says Half Her Cabinet Will Be ...,hillary clinton says half cabinet will women e...,"Women make up over 50 percent of this country,...",women make percent country grossly underrepres...,2016-04-26,2016,4,2220,196,0.196721,Fake News,1
3,Russian bombing of U.S.-backed forces being di...,russian bombing usbacked forces discussed high...,WASHINGTON (Reuters) - U.S. Defense Secretary ...,washington reuters us defense secretary jim ma...,2017-09-18,2017,9,627,56,0.061538,Factual News,0
4,Britain says window to restore Northern Irelan...,britain says window restore northern ireland d...,BELFAST (Reuters) - Northern Ireland s politic...,belfast reuters northern ireland political par...,2017-09-04,2017,9,745,67,0.046154,Factual News,0


In [19]:
# Final sanity checks before saving:
print('Final shape:', df.shape)
print()
print('Missing values per column:')
print(df.isnull().sum())
print()
print('Label balance:')
print(df['label'].value_counts())

Final shape: (198, 12)

Missing values per column:
title                0
clean_title          0
text                 0
clean_text           0
date                 0
year                 0
month                0
text_length          0
word_count           0
title_upper_ratio    0
fake_or_factual      0
label                0
dtype: int64

Label balance:
label
0    100
1     98
Name: count, dtype: int64


In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 198 entries, 0 to 197
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   title              198 non-null    object        
 1   clean_title        198 non-null    object        
 2   text               198 non-null    object        
 3   clean_text         198 non-null    object        
 4   date               198 non-null    datetime64[ns]
 5   year               198 non-null    int32         
 6   month              198 non-null    int32         
 7   text_length        198 non-null    int64         
 8   word_count         198 non-null    int64         
 9   title_upper_ratio  198 non-null    float64       
 10  fake_or_factual    198 non-null    object        
 11  label              198 non-null    int64         
dtypes: datetime64[ns](1), float64(1), int32(2), int64(3), object(5)
memory usage: 17.1+ KB


## 10. Save the Cleaned Dataset

Export the cleaned DataFrame to a new CSV, the same way `08-Data_Input_and_Output` covers
`df.to_csv(...)`. We keep the original raw file untouched and write this out as a new file,
so Phase 2 always starts from a known-good, cleaned dataset.

In [21]:
# index=False so pandas doesn't write its own row-number column into the CSV
df.to_csv('fake_news_data_clean.csv', index=False)

print('Saved cleaned dataset to fake_news_data_clean.csv')
print('Final shape:', df.shape)

Saved cleaned dataset to fake_news_data_clean.csv
Final shape: (198, 12)


## Summary — What We Did in Phase 1

1. Loaded the raw CSV and inspected it with `.head()`, `.info()`, `.shape`.
2. Checked for missing values (`isnull().sum()`) — found none, but verified for empty strings too.
3. Checked for duplicate rows — found none.
4. Stripped whitespace from all text columns.
5. Standardized the label column into a numeric `label` (1 = Fake, 0 = Factual).
6. Parsed the inconsistent `date` strings into real datetime objects, then extracted `year`/`month`.
7. **(NLP)** Cleaned the free-text `title`/`text` columns: lowercased, removed URLs/HTML/punctuation
   /digits/stopwords, producing `clean_title` and `clean_text`.
8. Engineered a few extra numeric features: `text_length`, `word_count`, `title_upper_ratio`.
9. Did a final review and saved everything to `fake_news_data_clean.csv`.

**Next up — Phase 2 (separate notebook):** vectorize `clean_text` (e.g. with `CountVectorizer` or
`TfidfVectorizer`), split into train/test sets, and train a classifier (Logistic Regression is a
natural first choice since we've already practiced it in the course) to predict `label`.